### A5.4.13. Build Minimal Machine Learning Library Architecture

**Problem:**

Build a minimal ML library that ties together a tensor class, a computational graph with autodiff, a kernel dispatch table, and a simple optimizer — demonstrating the end-to-end architecture of a machine learning framework.

**Explanation:**

A minimal ML library is a composition of the building blocks from the previous katas. Each component has one responsibility, and they connect through clean interfaces.

How to build it:

1. **Tensor**: stores data and shape (kata 01).
2. **Value with autodiff**: wraps scalar values, records operations, computes gradients via backprop (kata 06).
3. **Dispatch table**: maps operations to kernel implementations (kata 11).
4. **Parameter**: a `Value` that the optimizer updates.
5. **SGD optimizer**: takes a list of parameters, and on `step()`, updates each parameter by subtracting `learning_rate * gradient`.
6. Wire them together: build a forward pass using `Value` operations, call `backward()` to get gradients, call `optimizer.step()` to update parameters, then zero gradients.

**Example:**

Train a single linear neuron `y = w * x + b` to fit `y = 3x + 1` using mean squared error and SGD.

In [ ]:
class Value:
    def __init__(self, data, children=(), backward_fn=None):
        self.data = data
        self.grad = 0.0
        self.children = children
        self.backward_fn = backward_fn or (lambda: None)

    def __add__(self, other):
        out = Value(self.data + other.data, children=(self, other))
        def backward():
            self.grad += out.grad
            other.grad += out.grad
        out.backward_fn = backward
        return out

    def __mul__(self, other):
        out = Value(self.data * other.data, children=(self, other))
        def backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out.backward_fn = backward
        return out

    def __sub__(self, other):
        neg = Value(-1.0)
        return self + (neg * other)

    def __pow__(self, exponent):
        out = Value(self.data ** exponent, children=(self,))
        def backward():
            self.grad += exponent * (self.data ** (exponent - 1)) * out.grad
        out.backward_fn = backward
        return out

    def backward(self):
        topo_order = []
        visited = set()
        def build_topo(node):
            if node not in visited:
                visited.add(node)
                for child in node.children:
                    build_topo(child)
                topo_order.append(node)
        build_topo(self)
        self.grad = 1.0
        for node in reversed(topo_order):
            node.backward_fn()


class SGD:
    def __init__(self, parameters, learning_rate):
        self.parameters = parameters
        self.learning_rate = learning_rate

    def step(self):
        for param in self.parameters:
            param.data -= self.learning_rate * param.grad

    def zero_grad(self):
        for param in self.parameters:
            param.grad = 0.0


weight = Value(0.1)
bias = Value(0.0)
optimizer = SGD([weight, bias], learning_rate=0.01)

training_data = [(1.0, 4.0), (2.0, 7.0), (3.0, 10.0), (4.0, 13.0)]

for epoch in range(100):
    total_loss = Value(0.0)
    for x_val, y_true in training_data:
        x_input = Value(x_val)
        y_target = Value(y_true)
        y_pred = weight * x_input + bias
        loss = (y_pred - y_target) ** 2
        total_loss = total_loss + loss

    total_loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d} | Loss: {total_loss.data:.4f} | w={weight.data:.4f}, b={bias.data:.4f}")

print(f"\nLearned: y = {weight.data:.2f}x + {bias.data:.2f}")
print(f"Expected: y = 3.00x + 1.00")

**References:**

📘 Paszke, A. et al. (2019). *PyTorch: An Imperative Style, High-Performance Deep Learning Library* — NeurIPS

📘 Abadi, M. et al. (2016). *TensorFlow: A System for Large-Scale Machine Learning* — OSDI

---

[⬅️ Previous: Build Compilation Cache Key](./12_build_compilation_cache_key.ipynb)